# Simulated annealing vs gate-model QAOA

Two fundamentally different approaches to the same optimization problem:

1. **Simulated annealing** (D-Wave Ocean `neal`) — classical thermal
   search that mimics quantum annealing dynamics
2. **QAOA** (Qiskit) — a variational gate-model algorithm that prepares
   a parameterised superposition and optimises classically

We solve **MaxCut on C4** (4-cycle) with both and compare results.

This notebook is self-contained. It does not import `annealing_vs_gate.py`.

In [ ]:
import dimod
import neal
import numpy as np
import qiskit as qk
import scipy as scp

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0)]
N = 4
P = 2


def cut_size(bits):
    colors = [int(b) for b in bits[::-1]]
    return sum(colors[i] != colors[j] for i, j in EDGES)

## D-Wave Ocean: simulated annealing

The BQM encodes MaxCut as $E = -\sum_{(i,j)\in E} s_i s_j$. A
spin alignment ($s_i = s_j$) costs $-1$; anti-alignment gains $+1$.
`SimulatedAnnealingSampler` runs a classical annealing schedule.

In [ ]:
bqm = dimod.BinaryQuadraticModel("SPIN")
for i, j in EDGES:
    bqm.add_variable(i, 0.0)
    bqm.add_variable(j, 0.0)
    bqm.add_interaction(i, j, -1.0)

sampler = neal.SimulatedAnnealingSampler()
response = sampler.sample(bqm, num_reads=200, num_sweeps=500)
sample = response.first.sample
sa_bits = "".join(str(sample.get(i, 0)) for i in range(N))
sa_energy = response.first.energy
sa_cut = cut_size(sa_bits)

print(f"Best sample:  {sa_bits}")
print(f"BQM energy:   {sa_energy:+.2f}")
print(f"Cut value:    {sa_cut}")

## Qiskit QAOA: gate-model

The cost layer is CNOT–RZ–CNOT (ZZ interaction). The mixer is RX on
each qubit. COBYLA optimises the $2p$ parameters.

In [ ]:
def cost_layer(gamma):
    qc = qk.QuantumCircuit(N)
    for i, j in EDGES:
        qc.cx(i, j)
        qc.rz(2 * gamma, j)
        qc.cx(i, j)
    return qc


def mixer_layer(beta):
    qc = qk.QuantumCircuit(N)
    for q in range(N):
        qc.rx(2 * beta, q)
    return qc


def qaoa_circuit(params):
    qc = qk.QuantumCircuit(N)
    qc.h(range(N))
    for k in range(P):
        qc.compose(cost_layer(float(params[k])), inplace=True)
        qc.compose(mixer_layer(float(params[P + k])), inplace=True)
    return qc


def expected_cut(params):
    probs = qk.quantum_info.Statevector.from_instruction(
        qaoa_circuit(params)
    ).probabilities_dict()
    return sum(p * cut_size(b) for b, p in probs.items())


rng = np.random.default_rng(7)
guess = rng.uniform(0, np.pi, size=2 * P)
opt = scp.optimize.minimize(
    lambda p: -expected_cut(p),
    guess,
    method="COBYLA",
    options={"maxiter": 80, "rhobeg": 0.4},
)

probs = qk.quantum_info.Statevector.from_instruction(
    qaoa_circuit(opt.x)
).probabilities_dict()
qaoa_bits, qaoa_p = max(probs.items(), key=lambda kv: kv[1])
qaoa_exp = expected_cut(opt.x)
qaoa_cut = cut_size(qaoa_bits)

print(f"Most likely:  |{qaoa_bits}>")
print(f"Expected cut: {qaoa_exp:.3f}")
print(f"Cut value:    {qaoa_cut}")
print(f"Probability:  {qaoa_p:.3f}")

## Side-by-side comparison

In [ ]:
print(f"{'Method':<28} {'Sample':<12} {'Cut':>4}  Extra info")
print("-" * 65)
print(f"{'Simulated annealing (Ocean)':<28} {sa_bits:<12} {sa_cut:>4}  energy={sa_energy:+.1f}")
print(f"{'QAOA (Qiskit, p=2)':<28} |{qaoa_bits}>{'':<8} {qaoa_cut:>4}  P={qaoa_p:.3f}")
print()
print(f"Optimal cut on C4: {4}")